# Introduzione a PyTorch

PyTorch fornisce diversi moduli, di cui ``torch.nn`` è quello base per  la creazione dei modelli neurali, mentre i dati sono gestiti tramite ``torch.utils.data.DataSet`` che si occupa di caricare i dati dalle sorgenti e creare i tensori e ``torch.utils.data.DataLoader`` che si occupa di creare i batch e caricarli sul device selezionato per l'addestramento.
Esitono poi le librerie ``torchvision``, ``torchtext`` e ``torchaudio`` che forniscono data set specifici e funzioni di gestione per i tre domini di riferimento.

Utilizzeremo il nostro modulo ``torchnn`` con la semplice API per automatizzare la procedura di addestramento e valutazione del modello.

In [ ]:
# Importiamo tutte le librerire necessarie
import torch
from torch import nn
from torch.utils.data import random_split
from torchvision import datasets
from torchvision.transforms import v2
import os
import matplotlib.pyplot as plt
from torchsummary import summary
from torchnn import *

Il codice seguente serve a creare una semplice struttura di cartelle a partire da una root directory di vostra scelta, all'interno della quale vengono create due crtelle ``data`` per i data set e ``models`` per salvare i modelli.

In [ ]:
# root dei percorsi -- da impostare a seconda delle proprie esigenze
root = "/home/rpirrone/src"
os.makedirs(root, exist_ok=True)

# percorso dei dati
data_path = os.path.join(root,'data')

# lo creiamo la prima volta
if not os.path.exists(data_path):
    os.mkdir(data_path)

# Analogamente per il percorso dei modelli
model_path = os.path.join(root,'models')

if not os.path.exists(model_path):
    os.mkdir(model_path)


In [ ]:
# Acquisiamo il device su cui effettueremo il training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

Useremo il database ``FashionMNIST``per il nostro esempio: si tratta di 70000 immagini di dimensione 28x28, a livelli di grigio, che ritraggono capi di abbigliamento e sono associate a delle etichette testuali ch riportano il tipo di capo.

Abbiamo a che fare quindi con un problema di classificazione multiclasse di immagini che affronteremo con una rete neurale convoluizonale.

Il data set è suddiviso in un training set da 60000 immagini e un test set da 10000 immagini. Eseguiremo il download direttamente tramite ``torch`` che genererà un ``DataSet``.

In [ ]:
# Definiamo un set di trasformazioni per implementare la data augmentation
transforms = v2.Compose([
    v2.RandomAffine(degrees=(-20, 20), translate=(0.15, 0.15)), # rotazione random di un angolo tra -20° e +20°
    v2.RandomHorizontalFlip(p=0.5),                             # flip orizzontale con probabilità del 50%
    v2.ToImage(),                                               # Composizione di trasformazioni
    v2.ToDtype(torch.float32, scale=True)                       # equivalente a ToTensor e raccomandata dalla documentaizone
])

tensors = v2.Compose([
    v2.ToImage(),                                               # Composizione di trasformazioni
    v2.ToDtype(torch.float32, scale=True)])                     # equivalente a ToTensor e raccomandata dalla documentaizone

In [ ]:
# flag per forzare il download del data set solo la prima volta
download = not os.path.exists(os.path.join(data_path,'FashionMNIST'))

# Download del training set da un open data set.
training_data = datasets.FashionMNIST(
    root=data_path,
    train=True,         # esegue il download dalla cartella/archivio dei dati di train del data set originale
    download=download,
    transform=transforms,
)

# Download del training set da un open test set.
test = datasets.FashionMNIST(
    root=data_path,
    train=False,
    download=download,
    transform=tensors
)

# eseguiamo lo split 90% - 10% dei dati di train per creare il validation set
test_size = int(0.9 * len(test))
val_size = int(0.1 * len(test))

# spostiamo i dati sulla GPU prima dello split così gli indici dei due split fanno riferimento già a tensori sul device
training_data.data.to(device)
test.data.to(device)

test_data, val_data = random_split(test, [test_size, val_size])

print(f"Campioni nel training set: {len(training_data)}\
        \nCampioni nel validation set: {len(val_data)}\
        \nCampioni nel test set: {len(test_data)}")

### Tensori

Nel codice precedente abbiamo visto che la trasformazione usata nella creazione dei data set è quella tipica, ovvero la chiamata di ``torchvision.ToTensor()`` che converte un'immagine di dimensioni HxWxC in un tensore di dimensioni CxHxW. Le trasformazioni possono essere anche definite dall'utente attraverso funzioni lambda.

I tensori in Pytorch sono implementati in maniera molto simile agli array in ``numpy``: essi hanno diversi tipi dei dati, infatti la classe base ``torch.Tensor`` è il default che ha un ``dtype=torch.float32`` ed è un alias per ``torch.FloatTensor``.
Pytorch usa i tensori per implementare tutta l'algebra lineare, le operazioni di campionamento, e quelle che guidano la propagazione dei gradienti durante l'addestramento.

I tensori possono essere creati da specifiche operazioni che danno come risultato un tensore, come ``torch.randint()`` che vedremo tra poco e che genera un tensore di numeri interi casuali di data dimensione, oppure esplicitamente con ``torch.tensor()`` che crea un tensore da una struttura di tipo array.

### Dataset
Un ``Dataset`` è in genere implementato come una ``map`` Python, quindi con accesso a indice, ma può essere anche creato come sottoclasse di ``IterableDataset`` come un iteratore.

Diamo un'occhiata al nostro data set, ricordandoci che è implementato con accesso ad indice.

In [ ]:
labels_map = {
    0: "T-Shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}
figure = plt.figure(figsize=(4, 4))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    # Genero un tensore casuale composto da un solo elemento nel range della lunghezza del data set
    # e lo converto in numero usando il metodo Tensor.item()
    sample_idx = int(torch.randint(len(training_data), size=(1,)).item())
    img, label = training_data[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()

In [ ]:
# Creiamo i data loaders che saranno gli iterabili che generano i batch di
# addestramento e/o test
train_dataloader, val_dataloader, test_dataloader = make_dataloaders(training_data,
                                                                     val_data,
                                                                     test_data)

In [ ]:
#  Creiamo il modello
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.deep_conv_stack = nn.Sequential(
            nn.Conv2d(1, 64, 3),        # Shape: (N, 1, 28, 28)   --> (N, 64, 26, 26) a causa del kernel 3x3
            nn.ReLU(),
            nn.Conv2d(64, 128, 3),      # Shape: (N, 64, 26, 26)  --> (N, 128, 24, 24)
            nn.ReLU(),
            nn.Conv2d(128, 256, 3),     # Shape: (N, 128, 24, 24) --> (N, 256, 22, 22)
            nn.ReLU(),
            nn.Conv2d(256, 64, 3),      # Shape: (N, 256, 22, 22) --> (N, 64, 20, 20)
            nn.ReLU(),
            nn.AvgPool2d(2),            # Shape: (N, 64, 20, 20)  --> (N, 64, 10, 10) a causa della divisione per due
            nn.Conv2d(64, 32, 3),       # Shape: (N, 64, 10, 10)  --> (N, 32, 8, 8)
            nn.ReLU(),
            nn.AvgPool2d(2),            # Shape: (N, 32, 8, 8)    --> (N, 32, 4, 4)
            nn.Flatten(1,-1),           # Shape: (N, 32, 4, 4)    --> (N, 512) il flatten moltiplica tutte le dimensioni tra loro
            nn.Linear(512, 10)          # Shape: (N, 512)         --> (N, 10) strato denso di calcolo dei logits
        )

    def forward(self, x):
        # Calcolo della rete in avanti
        logits = self.deep_conv_stack(x)                # parte profonda della rete che estrae i logits
        pred_probab = nn.LogSoftmax(dim=1)(logits)      # uscita esplicita con log-softmax per la predizione delle etichette
        return pred_probab

# Il metodo to() sposta il modello sul device selezionato
model = NeuralNetwork().to(device)

# Riportiamo il sommario del modello con torchinfo.summary()
summary(model, input_size=(1,28,28))

In [ ]:
loss_fn = nn.NLLLoss()               # Negative log-likelihood

optimizer = torch.optim.SGD(
                            model.parameters(),
                            lr=config['learning_rate'],
                            momentum=config['momentum'],
                            nesterov=config['nesterov']
                            )

In [ ]:
# Creiamo la callback di early stopping con i oarametri di deafault della configurazione da passare al nostro metodo di addestramento
early_stopping = EarlyStopping()

# Routine di addestramento
train_loss, validation_loss, test_loss, accuracy, metrics = train_test(model,
                                                              optimizer,
                                                              device,
                                                              train_dataloader,
                                                              test_dataloader,
                                                              train_loss_fn=loss_fn,
                                                              test_loss_fn=loss_fn,
                                                              early_stopping=early_stopping,
                                                              val_dataloader=val_dataloader)

In [ ]:
displayLosses(train_loss, test_loss, validation_loss)

displayMetrics(accuracy, metrics)

## Salvataggio e caricamento del modello

Come già sappiamo, quando l'addestramento si interrompe a causa di un early stoppping ovvero perché è necessario frazionarlo per gestire le risorse limitate della GPU, ma anche semplicemente alla sua fine naturale il modello viene salvato per poterlo successivamente ricaricare a fini di fine tuning o di predizione. Si usa chiamare questa azione _model checkpoint_.

Anche questa operazione andrà codificata creando delle semplici API basate sulle primitive di ``Torch`` denominate ``torch.save()`` e ``torch.load()`` le quali salvano un qualunque ``dict`` in forma serializzata utilizzando il modulo nativo ``Python`` denominato ``pickle``.

Creeremo due semplici wrapper per le operazioni di salvataggio e caricamento del model checkpoint.

In [ ]:
# utilizziamo le API per salvare e ricaricare il modello
save_model(model,
           optimizer,
           len(train_loss),
           train_loss,
           validation_loss,
           test_loss,
           accuracy,
           metrics,
           os.path.join(model_path,'first_nn_torch.pth'))

mod = NeuralNetwork()
opt = torch.optim.SGD(mod.parameters(),
                            lr=config['learning_rate'],
                            momentum=config['momentum'],
                            nesterov=config['nesterov']
                            )

(mod, opt, model_checkpoint) = load_model(os.path.join(model_path,'first_nn_torch.pth'),
                                          mod,
                                          opt,
                                          device=device)

In [ ]:
# Vogliamo riaddestrare il modello dopo l'early stopping, per cui
# inseriamo uno scheduler che faccia decrescere il learning rate ad ogni epoca
# come ulteriore regolarizzazione
scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.9)

# Usiamo lo scheduler al posto dell'ottimizzatore nell'addestramento
tr_loss, val_loss, t_loss, new_accuracy, new_metrics = train_test(mod,
                                                                  opt, 
                                                                  device, 
                                                                  train_dataloader, 
                                                                  test_dataloader,
                                                                  epochs=5,
                                                                  train_loss_fn=loss_fn,
                                                                  test_loss_fn=loss_fn,
                                                                  scheduler=scheduler, 
                                                                  val_dataloader=val_dataloader)

In [ ]:
# concateniamo le liste delle loss e delle metriche relative ai due addestramenti e facciamo il plot
model_checkpoint['accuracy'].extend(new_accuracy)
model_checkpoint['training_loss'].extend(tr_loss)
model_checkpoint['validation_loss'].extend(val_loss)
model_checkpoint['test_loss'].extend(t_loss)

extended_metrics = {}
for label, metric in new_metrics.items():
    model_checkpoint[label].extend(metric)
    extended_metrics[label] = model_checkpoint[label]

displayLosses(model_checkpoint['training_loss'], model_checkpoint['test_loss'], model_checkpoint['validation_loss'])
displayMetrics(model_checkpoint['accuracy'], extended_metrics)

